# Search-05b : Natural Evolution Strategies en C#

**Navigation** : [Version Python](Search-05b-NaturalEvolutionStrategies.ipynb) · [Fondations](README.md)

**Objectifs** : dériver les gradients de recherche et naturels, comparer xNES, SNES et CMA-ES sur un ellipsoïde tourné, et comprendre le lien au premier ordre. **Prérequis** : probabilités, matrices et optimisation. **Durée** : 2 h.

## Préparation

Depuis la racine du dépôt, initialiser le sous-module et construire la dépendance exacte avant d'ouvrir ce carnet :

```bash
git submodule update --init --recursive MyIA.AI.Notebooks/Search/MetaGeneticSharp
dotnet build MyIA.AI.Notebooks/Search/MetaGeneticSharp/src/MetaGeneticSharp.Domain/MetaGeneticSharp.Domain.csproj -c Release
```

.NET 9 et le noyau .NET Interactive sont requis. Le gitlink fixe MetaGeneticSharp à `824c54e142efa6ae434890fc2ae83a3515aab1e3` et son sous-module GeneticSharp ; aucune clé ni GPU n'est nécessaire. Exécuter avec le répertoire courant de ce carnet. Les graphiques sont des SVG statiques.

In [1]:
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/GeneticSharp.Infrastructure.Framework.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/GeneticSharp.Domain.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/MetaGeneticSharp.Infrastructure.dll"
#r "../MetaGeneticSharp/src/MetaGeneticSharp.Domain/bin/Release/net9.0/MetaGeneticSharp.Domain.dll"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text;
using System.Net;
using System.Globalization;
using Microsoft.DotNet.Interactive.Formatting;
using MetaGeneticSharp;
using M = MetaGeneticSharp.NaturalEvolutionStrategyMath;
Console.WriteLine($"MetaGeneticSharp {typeof(NesDistribution).Assembly.GetName().Version}, GeneticSharp {typeof(GeneticSharp.IRandomization).Assembly.GetName().Version}");

The below script needs to be able to find the current output cell; this is an easy method to get it.

MetaGeneticSharp 1.0.0.0, GeneticSharp 3.1.4.0


## 1. Gradient de recherche et gradient naturel

Pour maximiser $E[-X^2]$, le gradient ordinaire estime les dérivées du log de la densité. Une mise à jour additive de $\sigma$ peut sortir du domaine positif. Le gradient naturel emploie des rangs et multiplie l'échelle par une exponentielle. Les médianes ne concernent que les trajectoires encore valides ; le nombre d'échecs est présenté séparément.

In [2]:
double Normal(Random r) => Math.Sqrt(-2*Math.Log(1-r.NextDouble()))*Math.Cos(2*Math.PI*r.NextDouble());
double[] Utilities(int n) { var a=Enumerable.Range(1,n).Select(k=>Math.Max(0,Math.Log(n/2.0+1)-Math.Log(k))).ToArray(); var sum=a.Sum(); return a.Select(x=>x/sum-1.0/n).ToArray(); }
int Lambda(int d)=>4+(int)Math.Floor(3*Math.Log(d));
(double[],double[]) OneD(int seed, bool natural) {
 int count=natural?200:1000, n=10; var mu=1.0; var sigma=1.0; var r=new Random(seed); var ms=Enumerable.Repeat(double.NaN,count).ToArray(); var ss=(double[])ms.Clone(); var u=Utilities(n);
 for(int g=0;g<count;g++) { var z=Enumerable.Range(0,n).Select(_=>Normal(r)).ToArray(); var x=z.Select(v=>mu+sigma*v).ToArray();
  if(natural) { var ranked=Enumerable.Range(0,n).OrderBy(i=>x[i]*x[i]).ToArray(); double gm=0,gs=0; for(int k=0;k<n;k++){double v=z[ranked[k]];gm+=u[k]*v;gs+=u[k]*(v*v-1);} mu+=sigma*gm; sigma*=Math.Exp(M.DefaultScaleLearningRate(NesCovarianceModel.Separable,1)*gs/2); }
  else { double gm=0,gs=0;for(int k=0;k<n;k++){double f=-x[k]*x[k];gm+=f*z[k]/sigma;gs+=f*(z[k]*z[k]-1)/sigma;} mu+=.01*gm/n;sigma+=.01*gs/n; }
  if(!double.IsFinite(mu)||!double.IsFinite(sigma)||sigma<=0) break; ms[g]=Math.Abs(mu); ss[g]=sigma;
 } return (ms,ss);
}
double Median(IEnumerable<double> xs) { var a=xs.Where(double.IsFinite).OrderBy(x=>x).ToArray(); return a.Length==0?double.NaN:(a[(a.Length-1)/2]+a[a.Length/2])/2; }
var simple=Enumerable.Range(0,300).Select(i=>OneD(i,false)).ToArray(); var naturel=Enumerable.Range(0,300).Select(i=>OneD(i,true)).ToArray();
foreach(var item in new[]{("simple",simple), ("naturel",naturel)}) { Console.WriteLine(item.Item1); foreach(var g in new[]{0,10,50,100,item.Item2[0].Item1.Length-1}.Distinct()) Console.WriteLine($"g={g}, échecs={item.Item2.Count(r=>!double.IsFinite(r.Item2[g]))}/300, médiane |mu|={Median(item.Item2.Select(r=>r.Item1[g])):G4}, sigma={Median(item.Item2.Select(r=>r.Item2[g])):G4}"); }

simple


g=0, échecs=0/300, médiane |mu|=0.9847, sigma=0.9931


g=10, échecs=0/300, médiane |mu|=0.8016, sigma=0.813


g=50, échecs=0/300, médiane |mu|=0.3562, sigma=0.3643


g=100, échecs=0/300, médiane |mu|=0.1295, sigma=0.1337


g=999, échecs=97/300, médiane |mu|=1.559E-09, sigma=1.58E-09


... (2 outputs left out)

g=10, échecs=0/300, médiane |mu|=0.0222, sigma=0.1032


g=50, échecs=0/300, médiane |mu|=2.83E-06, sigma=1.211E-05


g=100, échecs=0/300, médiane |mu|=3.434E-11, sigma=1.479E-10


g=199, échecs=0/300, médiane |mu|=4.655E-21, sigma=2.237E-20


Les échecs correspondent à une distribution invalide, pas à une valeur optimale. La mise en forme par rang rend le pas naturel insensible à toute transformation strictement croissante de l'objectif, sous réserve de conserver les mêmes classements.

In [3]:
var u10=Utilities(10); Console.WriteLine("Utilités : "+string.Join(", ",u10.Select(x=>x.ToString("F4")))+ $" ; somme={u10.Sum():G3}");
if(Math.Abs(u10.Sum())>1e-12) throw new Exception("Somme des utilités incorrecte");
foreach(int d in new[]{1,2,5,10,20,40}) Console.WriteLine($"d={d}, lambda={Lambda(d)}, xNES={M.DefaultShapeLearningRate(d):F5}, SNES={M.DefaultScaleLearningRate(NesCovarianceModel.Separable,d):F5}");
foreach(var name in new[]{"ExponentialNaturalEvolutionStrategy","SeparableNaturalEvolutionStrategy"}) Console.WriteLine($"Factory {name} : {MetaHeuristicsService.CreateMetaHeuristicByName(name).GetType().Name}");

Utilités : 0.3295, 0.1634, 0.0662, -0.0028, -0.0563, -0.1000, -0.1000, -0.1000, -0.1000, -0.1000 ; somme=-1.94E-16


d=1, lambda=4, xNES=1.80000, SNES=0.60000


d=2, lambda=6, xNES=0.78343, SNES=0.52229


d=5, lambda=8, xNES=0.24737, SNES=0.41228


d=10, lambda=10, xNES=0.10061, SNES=0.33536


d=20, lambda=12, xNES=0.04022, SNES=0.26814


d=40, lambda=15, xNES=0.01586, SNES=0.21152


Factory ExponentialNaturalEvolutionStrategy : NaturalEvolutionStrategyMetaHeuristic


Factory SeparableNaturalEvolutionStrategy : NaturalEvolutionStrategyMetaHeuristic


Les utilités centrées et les taux sont ceux de la table 1. Le moteur réel reçoit des échantillons triés du meilleur au moins bon. Le calcul explicite suivant est un oracle indépendant pour un pas SNES.

In [4]:
var anchors=new List<double[]>{new[]{1d,0d},new[]{-1d,0d},new[]{0d,1d},new[]{0d,-1d}};
var ranked=new List<double[]>{new[]{.1,.1},new[]{.3,.1},new[]{-.4,-.2},new[]{1d,1d}};
var init=NesDistribution.Fit(NesCovarianceModel.Separable,anchors); var eta=M.DefaultScaleLearningRate(NesCovarianceModel.Separable,2); var step=init.Update(ranked,1,eta,M.DefaultShapeLearningRate(2)); var weights=Utilities(ranked.Count);
for(int j=0;j<2;j++){double gm=0,gs=0;for(int k=0;k<ranked.Count;k++){double z=(ranked[k][j]-init.Mean[j])/init.Scales[j];gm+=weights[k]*z;gs+=weights[k]*(z*z-1);} double mean=init.Mean[j]+init.Scales[j]*gm, scale=init.Scales[j]*Math.Exp(eta*gs/2); Console.WriteLine($"coord {j}: écart moyenne={Math.Abs(mean-step.Mean[j]):G3}, écart échelle={Math.Abs(scale-step.Scales[j]):G3}"); if(Math.Abs(mean-step.Mean[j])+Math.Abs(scale-step.Scales[j])>1e-12) throw new Exception("Oracle SNES");}
var full=NesDistribution.Fit(NesCovarianceModel.Full,anchors);double rate=M.DefaultShapeLearningRate(2);var fullStep=full.Update(ranked,1,rate,rate);var gm2=new double[2];var matrix=new double[2,2];for(int k=0;k<ranked.Count;k++){var z=full.ToNatural(ranked[k]);for(int a=0;a<2;a++){gm2[a]+=weights[k]*z[a];for(int b=0;b<2;b++)matrix[a,b]+=weights[k]*(z[a]*z[b]-(a==b?1:0));}}double trace=(matrix[0,0]+matrix[1,1])/2;var shapeArg=new double[2,2];for(int a=0;a<2;a++)for(int b=0;b<2;b++)shapeArg[a,b]=rate/2*(matrix[a,b]-(a==b?trace:0));var expectShape=M.MatrixExponential(shapeArg);double maxError=0;for(int a=0;a<2;a++){maxError=Math.Max(maxError,Math.Abs(fullStep.Mean[a]-(full.Mean[a]+full.Sigma*gm2[a])));for(int b=0;b<2;b++)maxError=Math.Max(maxError,Math.Abs(fullStep.Shape[a,b]-expectShape[a,b]));}maxError=Math.Max(maxError,Math.Abs(fullStep.Sigma-full.Sigma*Math.Exp(rate*trace/2)));Console.WriteLine($"Oracle explicite xNES : écart maximal={maxError:G3}");if(maxError>1e-12)throw new Exception("Oracle xNES");

coord 0: écart moyenne=0, écart échelle=0


coord 1: écart moyenne=0, écart échelle=0


Oracle explicite xNES : écart maximal=0


### Exercice 1 — Utilités linéaires

Construire des poids décroissants centrés, puis comparer leurs trajectoires aux poids logarithmiques. Le squelette reste à compléter.

In [5]:
double[] LinearUtilities(int lambda) { // TODO : poids décroissants de somme nulle
 return null;
}
var linear=LinearUtilities(10);
if(linear==null) Console.WriteLine("Exercice 1 à compléter : utilités linéaires");
else {bool valid=linear.Length==10 && linear.All(double.IsFinite) && Math.Abs(linear.Sum())<1e-10 && linear.Zip(linear.Skip(1),(a,b)=>a>=b).All(v=>v) && linear[0]>linear[9]; Console.WriteLine($"Longueur, valeurs finies, somme nulle et ordre décroissant : {valid}"); if(!valid) throw new Exception("Corriger les utilités linéaires");}

Exercice 1 à compléter : utilités linéaires


## 2. Banc d'optimisation

On minimise un ellipsoïde translaté : son optimum n'est pas le point de départ $(3,\ldots,3)$. Les points symétriques imposent exactement moyenne 3 et écart-type 1 à `Fit`. La rotation est construite par rotations planes avec la graine 2014 ; ce n'est pas la matrice NumPy de la version Python. Le compteur comprend toutes les évaluations et le dernier lot peut dépasser le budget nominal.

In [6]:
double[,] Rotation(int d,int seed){var r=new Random(seed);var q=M.Identity(d);for(int k=0;k<d*4;k++){int a=r.Next(d),b=(a+1+r.Next(d-1))%d;double t=2*Math.PI*r.NextDouble(),c=Math.Cos(t),s=Math.Sin(t);for(int i=0;i<d;i++){double x=q[i,a],y=q[i,b];q[i,a]=c*x-s*y;q[i,b]=s*x+c*y;}}return q;}
double Objective(double[] x,double[,] q,bool rotated){int d=x.Length;double result=0;for(int i=0;i<d;i++){double y=0;for(int j=0;j<d;j++)y+=(rotated?q[j,i]:(i==j?1:0))*(x[j]-.7);result+=Math.Pow(1e6,(double)i/(d-1))*y*y;}return result;}
List<double[]> Anchors(int d){var a=new List<double[]>();for(int i=0;i<d;i++)foreach(int sign in new[]{-1,1}){var x=Enumerable.Repeat(3d,d).ToArray();x[i]+=sign*Math.Sqrt(d);a.Add(x);}return a;}
record Run(int Evals,double Best,List<(int,double)> Trace);
Run Optimize(string method,int d,int seed,int budget,Func<double[],double> f,double target=1e-8){
 var r=new Random(seed); int n=Lambda(d),evals=0; double best=double.PositiveInfinity; var trace=new List<(int,double)>(); var model=method=="SNES"?NesCovarianceModel.Separable:NesCovarianceModel.Full; var dist=NesDistribution.Fit(model,Anchors(d)); var cma=CmaEsDistribution.Create(Enumerable.Repeat(3d,d).ToArray(),1,M.Identity(d));
 while(evals<budget){var samples=new List<(double[],double)>();for(int k=0;k<n;k++){var z=Enumerable.Range(0,d).Select(_=>Normal(r)).ToArray();var x=method=="CMA-ES"?cma.FromNatural(z):dist.FromNatural(z);samples.Add((x,f(x)));}evals+=n;samples=samples.OrderBy(p=>p.Item2).ToList();best=Math.Min(best,samples[0].Item2);trace.Add((evals,best));if(best<=target)break;var xs=samples.Select(p=>p.Item1).ToList();if(method=="CMA-ES") cma=cma.UpdateWithFitness(xs,samples.Select(p=>-p.Item2).ToArray()); else dist=dist.Update(xs,1,M.DefaultScaleLearningRate(model,d),M.DefaultShapeLearningRate(d)); }
 return new Run(evals,best,trace);
}
var rot=Rotation(10,2014);double orth=0;for(int i=0;i<10;i++)for(int j=0;j<10;j++){double v=0;for(int k=0;k<10;k++)v+=rot[k,i]*rot[k,j];orth=Math.Max(orth,Math.Abs(v-(i==j?1:0)));}Console.WriteLine($"Erreur orthogonalité maximale={orth:G3} ; condition théorique préservée=1e6 ; f(départ)={Objective(Enumerable.Repeat(3d,10).ToArray(),rot,true):G5}");if(orth>1e-12)throw new Exception("Rotation non orthogonale");
var a10=NesDistribution.Fit(NesCovarianceModel.Full,Anchors(10)); Console.WriteLine($"Initialisation : moyenne={a10.Mean[0]}, sigma={a10.Sigma}");

Erreur orthogonalité maximale=6.66E-16 ; condition théorique préservée=1e6 ; f(départ)=4.4952E+06


Initialisation : moyenne=3, sigma=1


La transformation monotone conserve les rangs. Les deux runs ci-dessous utilisent le même flux aléatoire et le même budget ; l'écart est calculé après retour à l'échelle initiale.

In [7]:
var q3=Rotation(3,2014); Func<double[],double> f3=x=>Objective(x,q3,true); Func<double[],double> g3=x=>Math.Pow(Math.Log(1+f3(x)),3);var fRun=Optimize("xNES",3,5,900,f3,0);var gRun=Optimize("xNES",3,5,900,g3,0);double diff=fRun.Trace.Zip(gRun.Trace,(a,b)=>Math.Abs(a.Item2-(Math.Exp(Math.Pow(b.Item2,1.0/3))-1))).Max();Console.WriteLine($"f : {fRun.Evals} évaluations, g : {gRun.Evals} évaluations ; écart maximal ramené à f={diff:G5}");
double relative=fRun.Trace.Zip(gRun.Trace,(a,b)=>Math.Abs(a.Item2-(Math.Exp(Math.Pow(b.Item2,1.0/3))-1))/Math.Max(1,Math.Abs(a.Item2))).Max();if(fRun.Evals!=gRun.Evals||relative>1e-10)throw new Exception("Invariance monotone non vérifiée");Console.WriteLine($"Écart relatif maximal={relative:G4} ; invariance vérifiée");

f : 903 évaluations, g : 903 évaluations ; écart maximal ramené à f=3.7253E-08


Écart relatif maximal=2.8E-15 ; invariance vérifiée


### Exercice 2 — Coût en dimension

Mesurer les temps des mises à jour pour les dimensions 10, 40 et 100 sur les mêmes budgets. Comparer les coûts observés et ceux de la matrice pleine et de la diagonale.

In [8]:
int[] dimensions={10,40,100}; string[] methods={"xNES","SNES"};
Dictionary<(int,string),double> MeasureCosts() { // TODO : mesurer chaque paire dimension/méthode
 return null;
}
var costs=MeasureCosts();
if(costs==null) Console.WriteLine("Exercice 2 à compléter : chronométrage des six couples dimension/méthode");
else {bool valid=costs.Count==6 && dimensions.All(d=>methods.All(m=>costs.TryGetValue((d,m),out var value)&&double.IsFinite(value)&&value>=0));Console.WriteLine($"Six mesures finies et non négatives : {valid}");if(!valid)throw new Exception("Corriger la structure des mesures");}

Exercice 2 à compléter : chronométrage des six couples dimension/méthode


Le banc utilise quatre graines pour chaque méthode et chaque orientation. Le moteur CMA-ES maximise une fitness : on lui transmet donc $-f$ après tri croissant de $f$. Il s'agit de l'implémentation MGS, pas de la bibliothèque Python `cma` ; les RNG diffèrent aussi. Les résultats individuels ne sont pas supposés identiques.

In [9]:
var results=new Dictionary<string,List<Run>>();foreach(bool rotated in new[]{false,true}) foreach(string method in new[]{"xNES","SNES","CMA-ES"}) {var label=(rotated?"tourné":"aligné")+" "+method;var runs=new List<Run>();foreach(int seed in new[]{0,1,7,42})runs.Add(Optimize(method,10,seed,40000,x=>Objective(x,rot,rotated)));results[label]=runs;Console.WriteLine($"{label}: succès={runs.Count(r=>r.Best<=1e-8)}/4, évaluations min/médiane/max={runs.Min(r=>r.Evals)}/{Median(runs.Select(r=>(double)r.Evals))}/{runs.Max(r=>r.Evals)}, meilleur f={runs.Min(r=>r.Best):G4}");}

aligné xNES: succès=4/4, évaluations min/médiane/max=8850/8960/9110, meilleur f=7.374E-09


aligné SNES: succès=4/4, évaluations min/médiane/max=2630/2720/2810, meilleur f=6.614E-09


aligné CMA-ES: succès=4/4, évaluations min/médiane/max=5770/5990/6270, meilleur f=4.335E-09


tourné xNES: succès=4/4, évaluations min/médiane/max=8820/8870/8910, meilleur f=6.897E-09


tourné SNES: succès=0/4, évaluations min/médiane/max=40000/40000/40000, meilleur f=412.4


tourné CMA-ES: succès=4/4, évaluations min/médiane/max=5770/5955/6030, meilleur f=5.043E-09


Les comptes indiquent les évaluations réellement consommées, y compris le dernier lot. Le classement empirique sur cet exemple ne prouve pas l'équivalence par rotation de manière générale.

In [10]:
record PlotSvg(string Content);
Formatter.Register<PlotSvg>((p,w)=>w.Write(p.Content),"text/html");
string Esc(string s)=>WebUtility.HtmlEncode(s);
string F(double x)=>x.ToString("F2",CultureInfo.InvariantCulture);
string Figure(){var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='900' height='400' viewBox='0 0 900 400'>");string[] colors={"#276fbf","#c14953","#21856b"};int panel=0;foreach(string orientation in new[]{"aligné","tourné"}){int left=55+panel*440;sb.Append($"<text x='{left}' y='25'>{Esc(orientation)}</text><path d='M{left} 40 V340 H{left+390}' fill='none' stroke='black'/><text x='{left+100}' y='390'>évaluations effectives</text><text x='{left}' y='355'>0</text><text x='{left+340}' y='355'>40000</text><text x='{left}' y='38'>log10 f</text>");foreach(int tick in new[]{-8,-4,0,4,8}) {double y=40+300*(8-tick)/18.0;sb.Append($"<text x='{left-30}' y='{F(y+4)}'>{tick}</text><path d='M{left-4} {F(y)} H{left}' stroke='black'/>");}foreach(int tick in new[]{10000,20000,30000}) {double x=left+390.0*tick/40000;sb.Append($"<text x='{F(x-14)}' y='355'>{tick/1000}k</text><path d='M{F(x)} 340 v4' stroke='black'/>");}int m=0;foreach(string method in new[]{"xNES","SNES","CMA-ES"}){var run=results[orientation+" "+method].OrderBy(r=>r.Evals).ThenBy(r=>r.Best).ElementAt(1);var pts=run.Trace.Where((p,i)=>i%10==0||i==run.Trace.Count-1).Select(p=>$"{F(left+390.0*p.Item1/40010)} {F(40+300*(8-Math.Clamp(Math.Log10(Math.Max(p.Item2,1e-12)),-10,8))/18)}");sb.Append($"<polyline fill='none' stroke='{colors[m]}' points='{string.Join(" ",pts)}'/><text x='{left+90*m}' y='375' fill='{colors[m]}'>{Esc(method)}</text>");m++;}sb.Append($"<path d='M{left} 306.67 H{left+390}' stroke='#777' stroke-dasharray='4 3'/><text x='{left+300}' y='302'>cible 1e-8</text>");panel++;}return sb.Append("</svg>").ToString();}
display(new PlotSvg(Figure()));
string GradientFigure(){var sb=new StringBuilder("<svg xmlns='http://www.w3.org/2000/svg' width='900' height='400' viewBox='0 0 900 400'>");int panel=0;foreach(var data in new[]{simple,naturel}){int left=55+panel*440, count=data[0].Item1.Length;sb.Append($"<text x='{left}' y='25'>{(panel==0?"gradient simple":"gradient naturel")}</text><path d='M{left} 40 V340 H{left+390}' fill='none' stroke='black'/><text x='{left}' y='38'>log10 valeur (0 à -22)</text><text x='{left+100}' y='390'>génération (0 à {count-1})</text>");for(int series=0;series<3;series++){var pts=new List<string>();for(int g=0;g<count;g+=Math.Max(1,count/200)){double value=series==0?Median(data.Select(r=>r.Item1[g])):series==1?Median(data.Select(r=>r.Item2[g])):data.Count(r=>!double.IsFinite(r.Item2[g]))/300.0;double y=series==2?340-300*value:40+300*Math.Clamp(-Math.Log10(Math.Max(value,1e-22))/22,0,1);pts.Add($"{F(left+390.0*g/(count-1))} {F(y)}");}var color=series==0?"#276fbf":series==1?"#c14953":"#555555";sb.Append($"<polyline fill='none' stroke='{color}' points='{string.Join(" ",pts)}'/>");}sb.Append($"<text x='{left}' y='370' fill='#276fbf'>|mu|</text><text x='{left+60}' y='370' fill='#c14953'>sigma</text><text x='{left+130}' y='370' fill='#555'>échecs : 0–100 %</text>");panel++;}return sb.Append("</svg>").ToString();}
display(new PlotSvg(GradientFigure()));

aligné évaluations effectives 0 40000 log10 f -8 -4 0 4 8 10k 20k 30k <polyline fill='none' stroke='#276fbf' points='55.10 72.80 56.07 94.65 57.05 94.65 58.02 97.60 59.00 97.60 59.97 97.60 60.95 107.08 61.92 108.58 62.90 116.95 63.87 116.95 64.85 116.95 65.82 116.95 66.79 120.21 67.77 125.69 68.74 126.75 69.72 130.70 70.69 130.70 71.67 131.04 72.64 131.04 73.62 134.48 74.59 136.36 75.57 137.93 76.54 139.70 77.52 142.39 78.49 143.42 79.47 146.87 80.44 149.06 81.42 152.32 82.39 158.46 83.37 167.98 84.34 167.98 85.31 167.98 86.29 174.42 87.26 176.49 88.24 176.49 89.21 177.13 90.19 179.53 91.16 182.15 92.14 188.21 93.11 188.21 94.09 188.21 95.06 190.37 96.04 194.39 97.01 196.00 97.99 203.92 98.96 203.92 99.94 205.54 100.91 205.54 101.89 205.89 102.86 211.80 103.84 211.80 104.81 218.58 105.78 218.58 106.76 222.06 107.73 222.06 108.71 222.06 109.68 226.50 110.66 228.95 111.63 233.47 112.61 233.47 113.58 235.01 114.56 236.78 115.53 238.64 116.51 243.24 117.48 250.74 118.46 250.74 119.43 250.74 120.41 253.59 121.38 256.73 122.36 259.33 123.33 263.30 124.31 263.30 125.28 267.21 126.25 268.57 127.23 269.96 128.20 273.58 129.18 273.58 130.15 279.04 131.13 279.73 132.10 281.01 133.08 282.53 134.05 283.64 135.03 288.43 136.00 291.45 136.98 291.45 137.95 291.45 138.93 300.74 139.90 300.74 140.88 300.74 141.75 306.95'/> xNES SNES <polyline fill='none' stroke='#21856b' points='55.10 66.73 56.07 90.94 57.05 99.41 58.02 105.20 59.00 105.20 59.97 112.39 60.95 112.39 61.92 119.79 62.90 119.79 63.87 122.11 64.85 123.93 65.82 126.04 66.79 127.60 67.77 128.08 68.74 128.47 69.72 129.07 70.69 129.51 71.67 130.02 72.64 131.03 73.62 132.33 74.59 133.51 75.57 134.14 76.54 134.66 77.52 135.13 78.49 135.71 79.47 137.50 80.44 137.83 81.42 139.50 82.39 141.98 83.37 144.97 84.34 145.41 85.31 146.34 86.29 147.47 87.26 149.78 88.24 153.07 89.21 156.17 90.19 162.53 91.16 166.52 92.14 167.43 93.11 168.09 94.09 168.33 95.06 168.40 96.04 168.46 97.01 168.60 97.99 169.08 98.96 170.22 99.94 171.75 100.91 177.83 101.89 181.60 102.86 191.47 103.84 201.51 104.81 215.02 105.78 229.26 106.76 234.88 107.73 245.28 108.71 259.49 109.68 273.21 110.66 282.88 111.63 290.15 112.41 308.21'/> CMA-ES cible 1e-8 tourné évaluations effectives 0 40000 log10 f -8 -4 0 4 8 10k 20k 30k <polyline fill='none' stroke='#276fbf' points='495.10 69.64 496.07 89.30 497.05 96.06 498.02 96.50 499.00 96.75 499.97 98.89 500.95 107.86 501.92 111.39 502.90 111.39 503.87 111.53 504.85 111.53 505.82 119.41 506.79 123.68 507.77 123.75 508.74 131.38 509.72 133.03 510.69 133.03 511.67 135.76 512.64 139.52 513.62 142.84 514.59 152.47 515.57 152.47 516.54 152.47 517.52 153.78 518.49 154.01 519.47 156.29 520.44 158.85 521.42 158.88 522.39 164.12 523.37 169.94 524.34 169.94 525.31 172.44 526.29 172.44 527.26 177.32 528.24 184.51 529.21 184.51 530.19 184.51 531.16 191.48 532.14 191.48 533.11 191.48 534.09 193.60 535.06 198.11 536.04 198.11 537.01 199.34 537.99 200.66 538.96 201.71 539.94 205.11 540.91 207.26 541.89 209.73 542.86 213.20 543.84 213.20 544.81 219.45 545.78 224.80 546.76 224.80 547.73 224.80 548.71 227.08 549.68 227.08 550.66 227.08 551.63 232.45 552.61 236.81 553.58 236.81 554.56 238.53 555.53 240.33 556.51 242.80 557.48 246.57 558.46 253.20 559.43 253.20 560.41 253.20 561.38 258.15 562.36 258.15 563.33 262.33 564.31 265.20 565.28 267.78 566.25 268.14 567.23 268.14 568.20 269.41 569.18 274.91 570.15 281.70 571.13 281.70 572.10 281.82 573.08 284.80 574.05 284.80 575.03 290.90 576.00 291.53 576.98 296.99 577.95 300.26 578.93 300.26 579.90 301.18 580.88 303.06 581.27 309.36'/> xNES <polyline fill='none' stroke='#c14953' points='495.10 70.67 496.07 92.98 497.05 92.98 498.02 92.98 499.00 92.98 499.97 97.12 500.95 101.92 501.92 104.99 502.90 107.35 503.87 108.24 504.85 110.50 505.82 110.74 506.79 111.73 507.77 112.01 508.74 113.76 509.72 115.29 510.69 115.69 511.67 116.47 512.64 117.04 513.62 117.65 514.59 118.01 515.57 118.46 516.54 118.84 517.52 119.30 518.49 119.56 519.47 119.75 520.44 119.88 521.42 

gradient simple log10 valeur (0 à -22) génération (0 à 999) <polyline fill='none' stroke='#276fbf' points='55.00 40.09 56.95 40.69 58.90 41.31 60.86 41.90 62.81 42.51 64.76 43.10 66.71 43.72 68.66 44.30 70.62 44.89 72.57 45.51 74.52 46.11 76.47 46.63 78.42 47.27 80.38 47.86 82.33 48.45 84.28 49.09 86.23 49.68 88.18 50.31 90.14 50.85 92.09 51.51 94.04 52.11 95.99 52.71 97.94 53.40 99.89 54.02 101.85 54.54 103.80 55.06 105.75 55.77 107.70 56.35 109.65 57.03 111.61 57.55 113.56 58.13 115.51 58.72 117.46 59.33 119.41 59.96 121.37 60.51 123.32 61.13 125.27 61.73 127.22 62.38 129.17 62.98 131.13 63.58 133.08 64.23 135.03 64.81 136.98 65.39 138.93 66.06 140.89 66.57 142.84 67.25 144.79 67.82 146.74 68.47 148.69 69.07 150.65 69.72 152.60 70.31 154.55 70.93 156.50 71.60 158.45 72.07 160.41 72.69 162.36 73.32 164.31 73.81 166.26 74.52 168.21 75.16 170.17 75.68 172.12 76.23 174.07 76.80 176.02 77.41 177.97 77.95 179.92 78.61 181.88 79.18 183.83 79.80 185.78 80.46 187.73 80.98 189.68 81.56 191.64 82.19 193.59 82.70 195.54 83.30 197.49 83.93 199.44 84.61 201.40 85.08 203.35 85.72 205.30 86.36 207.25 86.90 209.20 87.39 211.16 88.07 213.11 88.66 215.06 89.14 217.01 89.77 218.96 90.29 220.92 90.95 222.87 91.54 224.82 92.22 226.77 92.82 228.72 93.38 230.68 94.09 232.63 94.77 234.58 95.32 236.53 96.03 238.48 96.55 240.44 97.22 242.39 97.86 244.34 98.30 246.29 98.97 248.24 99.65 250.20 100.19 252.15 100.89 254.10 101.45 256.05 102.12 258.00 102.72 259.95 103.30 261.91 103.87 263.86 104.46 265.81 104.91 267.76 105.52 269.71 106.19 271.67 106.78 273.62 107.38 275.57 108.00 277.52 108.51 279.47 109.07 281.43 109.78 283.38 110.39 285.33 111.01 287.28 111.65 289.23 112.24 291.19 112.84 293.14 113.44 295.09 114.16 297.04 114.57 298.99 115.19 300.95 115.85 302.90 116.53 304.85 117.09 306.80 117.65 308.75 118.29 310.71 118.95 312.66 119.45 314.61 120.01 316.56 120.60 318.51 121.22 320.47 121.77 322.42 122.45 324.37 123.03 326.32 123.62 328.27 124.13 330.23 124.80 332.18 125.30 334.13 125.88 336.08 126.48 338.03 127.01 339.98 127.55 341.94 128.25 343.89 128.84 345.84 129.39 347.79 130.24 349.74 130.80 351.70 131.36 353.65 132.01 355.60 132.65 357.55 133.10 359.50 133.62 361.46 134.30 363.41 135.00 365.36 135.51 367.31 136.15 369.26 136.68 371.22 137.38 373.17 137.92 375.12 138.35 377.07 138.88 379.02 139.44 380.98 140.09 382.93 140.86 384.88 141.36 386.83 142.12 388.78 142.70 390.74 143.29 392.69 143.95 394.64 144.59 396.59 145.07 398.54 145.60 400.50 146.21 402.45 146.82 404.40 147.40 406.35 147.99 408.30 148.52 410.26 149.14 412.21 149.84 414.16 150.33 416.11 151.00 418.06 151.67 420.02 152.30 421.97 152.75 423.92 153.42 425.87 154.08 427.82 154.72 429.77 155.23 431.73 155.84 433.68 156.42 435.63 157.15 437.58 157.72 439.53 158.40 441.49 159.04 443.44 159.57'/><polyline fill='none' stroke='#c14953' points='55.00 40.04 56.95 40.62 58.90 41.23 60.86 41.80 62.81 42.40 64.76 43.08 66.71 43.65 68.66 44.25 70.62 44.83 72.57 45.51 74.52 45.98 76.47 46.59 78.42 47.18 80.38 47.75 82.33 48.35 84.28 48.93 86.23 49.53 88.18 50.20 90.14 50.76 92.09 51.40 94.04 51.91 95.99 52.64 97.94 53.35 99.89 53.96 101.85 54.46 103.80 55.12 105.75 55.69 107.70 56.36 109.65 57.02 111.61 57.54 113.56 58.07 115.51 58.72 117.46 59.24 119.41 60.24 121.37 60.76 123.32 61.32 125.27 61.88 127.22 62.30 129.17 63.07 131.13 63.62 133.08 64.37 135.03 64.99 136.98 65.49 138.93 66.06 140.89 66.79 142.84 67.37 144.79 68.09 146.74 68.83 148.69 69.66 150.65 70.31 152.60 70.82 154.55 71.51 156.50 71.91 158.45 72.82 160.41 73.34 162.36 73.80 164.31 74.56 166.26 75.09 168.21 75.74 170.17 76.41 172.12 76.81 174.07 77.24 176.02 77.79 177.97 78.64 179.92 79.06 181.88 79.62 183.83 80.38 185.78 80.81 187.73 81.57 189.68 82.11 191.64 82.79 193.59 83.48 195.54 83.95 197.49 84.76 199.44 85.00 201.40 85.73 203.35 86.19 205.30 86.76 207.25 87.39 209.20 87.74 211.16 88.42 213.11 89.04 215.06 89.62 217.01 90.23 218.96 91.02 220.92 91.43 222.87 92.00 224.82 92.86 226.77 93.30 228.72 93.79 230.68 94.74 232.63 95

Chaque courbe montre un run représentatif et le meilleur objectif cumulé. L'axe vertical est logarithmique ; la cible apparaît en pointillé.

## 3. Premier ordre et CMA-ES

Pour une matrice symétrique $G$, $\exp(\eta G)=I+\eta G+\frac12\eta^2G^2+O(\eta^3)$. Ce rang-$\mu$ isolé n'est pas le CMA-ES complet, qui inclut notamment des chemins d'évolution.

In [11]:
var rng5=new Random(42);int d5=5,n5=Lambda(d5);var w5=Utilities(n5);var G=new double[d5,d5];for(int k=0;k<n5;k++){var z=Enumerable.Range(0,d5).Select(_=>Normal(rng5)).ToArray();for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)G[i,j]+=w5[k]*(z[i]*z[j]-(i==j?1:0));}
var G2=M.Multiply(G,G);double quad=0;foreach(double x in G2)quad+=x*x;Console.WriteLine($"Coefficient quadratique prédit = {Math.Sqrt(quad)/2:G6}");foreach(double rate in new[]{.5,.1,.02,.004}){var A=new double[d5,d5];for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)A[i,j]=rate*G[i,j];var E=M.MatrixExponential(A);double err=0;for(int i=0;i<d5;i++)for(int j=0;j<d5;j++){double e=E[i,j]-(i==j?1:0)-A[i,j];err+=e*e;}err=Math.Sqrt(err);Console.WriteLine($"eta={rate}: erreur={err:G6}, erreur/eta²={err/(rate*rate):G6}");}
var small=M.MatrixExponential(new double[,]{{0,1},{-1,0}});if(Math.Abs(small[0,0]-Math.Cos(1))+Math.Abs(small[0,1]-Math.Sin(1))>1e-12)throw new Exception("Oracle exponentielle");double etaSmall=.004;var Atest=new double[d5,d5];for(int i=0;i<d5;i++)for(int j=0;j<d5;j++)Atest[i,j]=etaSmall*G[i,j];var Etest=M.MatrixExponential(Atest);double e2=0;for(int i=0;i<d5;i++)for(int j=0;j<d5;j++){double v=Etest[i,j]-(i==j?1:0)-Atest[i,j];e2+=v*v;}double ratio=Math.Sqrt(e2)/(etaSmall*etaSmall);if(Math.Abs(ratio-Math.Sqrt(quad)/2)>0.005)throw new Exception("Coefficient quadratique");Console.WriteLine("Oracle expm et coefficient quadratique vérifiés");

Coefficient quadratique prédit = 0.75651


eta=0.5: erreur=0.204404, erreur/eta²=0.817614


eta=0.1: erreur=0.00764198, erreur/eta²=0.764198


eta=0.02: erreur=0.000303148, erreur/eta²=0.75787


eta=0.004: erreur=1.21084E-05, erreur/eta²=0.756775


Oracle expm et coefficient quadratique vérifiés


L'erreur divisée par $\eta^2$ tend vers le coefficient quadratique affiché. La fonction exponentielle matricielle utilisée ici est celle du moteur MetaGeneticSharp.

### Exercice 3 — Rosenbrock tourné

Implémenter Rosenbrock, translater son optimum puis appliquer la rotation. Comparer les trois méthodes à budget égal.

In [12]:
double? Rosenbrock(double[] x) { // TODO : somme 100(x[i+1]-x[i]^2)^2+(1-x[i])^2
 return null;
}
double[] RotateRosenbrockInput(double[] x, double[,] q) { // TODO : appliquer q au vecteur, puis ajouter 1 pour translater l'optimum
 return null;
}
var trial=Rosenbrock(new[]{1d,1d}); var translated=RotateRosenbrockInput(new[]{0d,0d},M.Identity(2));
if(trial==null || translated==null) Console.WriteLine("Exercice 3 à compléter : Rosenbrock et transformation tournée");
else {var q=Rotation(2,2014);var transformed=RotateRosenbrockInput(new[]{.3,-.4},q);double[] expected={1+q[0,0]*.3-q[0,1]*.4,1+q[1,0]*.3-q[1,1]*.4};bool valid=trial.HasValue && Math.Abs(trial.Value)<1e-12 && Math.Abs(Rosenbrock(new[]{0d,0d}).GetValueOrDefault(double.NaN)-1)<1e-12 && Math.Abs(Rosenbrock(new[]{-1d,1d}).GetValueOrDefault(double.NaN)-4)<1e-12 && translated.Length==2 && translated.All(v=>Math.Abs(v-1)<1e-12) && transformed!=null && transformed.Length==2 && transformed.Zip(expected,(a,b)=>Math.Abs(a-b)<1e-12).All(v=>v);Console.WriteLine($"Fixtures Rosenbrock et transformation : {valid}");if(!valid)throw new Exception("Corriger Rosenbrock ou sa rotation");}

Exercice 3 à compléter : Rosenbrock et transformation tournée


## À retenir

| Question | Réponse |
|---|---|
| Pourquoi le gradient naturel ? | La géométrie de la distribution modifie l'échelle du pas. |
| Que préserve le classement ? | Toute transformation strictement croissante de l'objectif. |
| Que révèle une rotation ? | La sensibilité d'une covariance diagonale aux axes du problème. |
| Qu'apporte le développement ? | Le lien local entre exponentielle matricielle et mise à jour rang-$\mu$. |

## Vérifiez votre compréhension

Pourquoi une mise à jour additive de $\sigma$ peut-elle échouer ?
<details><summary>Réponse</summary>Elle peut rendre l'écart-type négatif ou non fini ; la distribution n'est alors plus valide.</details>

## Pour aller plus loin

Wierstra et al., *Natural Evolution Strategies*, JMLR 2014 ; [version Python](Search-05b-NaturalEvolutionStrategies.ipynb).